# 01 — Thu thập và khám phá dữ liệu

> **Bản tiếng Việt** của `kaggle/01_compare_with_description_run.ipynb`. Code và output giữ nguyên bản gốc; chỉ phần chữ giải thích và comment được dịch. Thuật ngữ kỹ thuật và tên cột giữ nguyên tiếng Anh.

## 1. Bối cảnh thu thập dữ liệu

- **Chủ đề:** anime và người dùng của MyAnimeList (MAL), nơi người dùng tự lập danh sách anime của mình và chấm điểm từ 1–10.
- **Đơn vị quan sát:** ba bảng; một dòng lần lượt là một anime, một user, và một cặp user–anime.
- **Nguồn:** [MyAnimeList Dataset trên Kaggle](https://www.kaggle.com/datasets/azathoth42/myanimelist), tác giả Matěj Račinský, 2018, doi: 10.34740/KAGGLE/DSV/45582.
- **Giấy phép:** Kaggle ghi "Database: Open Database, Contents: Database Contents". Tác giả cho phép dùng mà không cần xin phép hay trả phí, và yêu cầu trích dẫn như trên khi dùng dữ liệu trong ấn phẩm hoặc bài thuyết trình.
- **Cách thu thập:** crawl dữ liệu công khai từ MyAnimeList.net. MAL định danh user bằng username nên không thể duyệt tuần tự theo user; tác giả lấy username từ các thread diễn đàn watching-challenge năm 2015–2018 và từ khoảng 40k club đầu tiên trong tổng ~80k club.
- **Hạn chế đã biết:**
  - User đến từ diễn đàn và club, nên mẫu có thể nghiêng về người dùng tích cực và không đại diện cho mọi user của MAL.
  - Dữ liệu dừng lại vào khoảng năm 2018.
  - Với mỗi cặp user–anime chỉ giữ trạng thái hiện tại và lần cập nhật cuối, không có lịch sử thay đổi.
- **Phiên bản dùng:** 3 file raw (`AnimeList.csv`, `UserList.csv`, `UserAnimeList.csv`). Các bản filtered và cleaned do tác giả xử lý, chỉ dùng để so sánh.
- **Vì sao chọn dataset này:** **TODO (nhóm tự viết)**

**Yêu cầu của môn học:** dữ liệu công khai, có thật, truy vết được nguồn, ít nhất ~3,000 dòng và 10 cột, không phải dữ liệu tổng hợp.


In [1]:
from pathlib import Path
import sys

import pandas as pd

# Chạy được cả trên Kaggle lẫn trong repo (với dữ liệu đặt ở data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))

Python: 3.13.18 | pandas: 2.3.3
Number of CSV files: 9


## 2. Đọc dữ liệu và so sánh với mô tả

**Một dòng của `UserAnimeList.csv` nghĩa là gì:** một user và một anime trong danh sách của user đó. Bước này chỉ đọc 4 cột cần cho việc so sánh vì file nặng khoảng 5 GB.

| Cột | Ý nghĩa | Kiểu dữ liệu |
|---|---|---|
| `username` | Định danh user | Key |
| `anime_id` | Định danh anime | Key |
| `my_score` | Điểm user chấm, 1–10 theo mô tả | Ordinal (có thứ tự) |
| `my_status` | "Kệ" mà user đặt anime vào | Nominal (danh mục): các mã chỉ là nhãn nên tính trung bình là vô nghĩa |

Các mã `my_status` theo mô tả: 1 watching (đang xem), 2 completed (đã xem xong), 3 on hold (tạm dừng), 4 dropped (bỏ dở), 6 plan to watch (dự định xem); các giá trị khác không được giải thích.

Trước khi viết `assert`, ta in số thực tế ra cạnh số trong mô tả, vì một số khác biệt có thể là hợp lý.


In [2]:
# Safe load: chỉ ô trống mới là giá trị thiếu. pandas mặc định còn đọc username `null` thành thiếu
na_kwargs = dict(keep_default_na=False, na_values=[""])
anime_raw = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users_raw = pd.read_csv(paths["UserList.csv"], **na_kwargs)
lists_raw = pd.read_csv(
    paths["UserAnimeList.csv"], **na_kwargs,
    usecols=["username", "anime_id", "my_score", "my_status"],
    dtype={"username": "category", "anime_id": "int32",
           "my_score": "int32", "my_status": "int32"},
)

# So sánh với mô tả trước, chưa dùng assert
checks = {
    "records":       (len(lists_raw), 80_076_112),
    "records rated": (int((lists_raw.my_score > 0).sum()), 46_358_322),
    "users (lists)": (lists_raw.username.nunique(), 302_675),
    "users (table)": (users_raw.username.nunique(), 302_675),
    "anime (table)": (anime_raw.anime_id.nunique(), 14_478),
    "anime (lists)": (lists_raw.anime_id.nunique(), 14_478),
}
for name, (actual, expected) in checks.items():
    print(f"{name:15s} actual={actual:>12,} description={expected:>12,} diff={actual - expected:+,}")

print("anime_id unique:", anime_raw.anime_id.is_unique)
print("title unique   :", anime_raw.title.is_unique)
print(lists_raw.my_status.value_counts().sort_index())
print(lists_raw.my_score.value_counts().sort_index())

records         actual=  80,076,112 description=  80,076,112 diff=+0
records rated   actual=  46,358,322 description=  46,358,322 diff=+0
users (lists)   actual=     283,045 description=     302,675 diff=-19,630
users (table)   actual=     302,675 description=     302,675 diff=+0
anime (table)   actual=      14,478 description=      14,478 diff=+0
anime (lists)   actual=      14,478 description=      14,478 diff=+0
anime_id unique: True
title unique   : False
my_status
0          527
1      4057788
2     50445526
3      2985804
4      3194353
5            6
6     19392104
33           3
55           1
Name: count, dtype: int64
my_score
0     33717790
1       321270
2       374744
3       619587
4      1296354
5      2905183
6      5456518
7     10379184
8     11394082
9      7897871
10     5713529
Name: count, dtype: int64


**Bước này nằm ở đâu.** Bước 1 trong thứ tự khám phá (xem [docs/exploration_order.md](../docs/exploration_order.md)). Bản ghi này chứa lần chạy **thứ hai** của notebook 01, dùng safe load (chỉ ô trống mới là giá trị thiếu). Lần chạy đầu dùng mặc định của pandas và làm mất username `null`; nó được lưu trong lịch sử git và được so sánh với lần này ở mục 2.2 c. Các thống kê ở 2.2 a (điểm theo status) là kết quả trên dữ liệu raw và sẽ được kiểm tra lại sau khi làm sạch.


### 2.1 So sánh với mô tả

**Ta làm gì ở đây.** Tác giả dataset có viết một bản mô tả: mỗi file có bao nhiêu bản ghi, có những cột nào, và một cột được phép chứa những mã nào. Ta không tin ngay. Với mỗi điều tác giả nói, ta đếm lại đúng thứ đó trong file thật và ghi lại mọi chỗ file nói khác đi. *Khớp* nghĩa là ta đếm ra cùng con số. *Khác* nghĩa là ta đếm ra con số khác.

**Cách đọc file.** Chỉ **ô trống** mới được coi là thiếu (`keep_default_na=False, na_values=[""]`). Pandas mặc định còn coi các từ như `null` là thiếu, và điều đó làm mất một user có thật (xem 2.2 c).

**Khớp.**
- 80,076,112 bản ghi trong list table.
- **46,358,322 bản ghi có điểm.**
- 14,478 anime trong cả anime table và list table, và hai tập `anime_id` giống hệt nhau.
- **302,675 username trong user table, không có username nào lặp.**

**Khác với mô tả.**

| Mục | Mô tả | Thực tế | Chênh lệch | Giải thích | Trạng thái |
|---|---|---|---|---|---|
| Số username trong `UserAnimeList` | 302,675 | 283,045 | −19,630 | User có list rỗng thì không có dòng nào (xem 2.2 d) | Giải thích được 19,622 trong 19,630; 8 user chưa giải thích được |
| Các giá trị `my_status` | 1, 2, 3, 4, 6 | thêm 0, 5, 33, 55 | 537 dòng | Mô tả nói các giá trị khác là không rõ | Cần quyết định (xem 2.2 b) |

Cách đọc các khác biệt:
- **Username trong list table.** Mô tả nói có 302,675 user, nhưng chỉ 283,045 user có ít nhất một dòng. 19,630 user còn lại có trong user table nhưng không có dòng nào trong list table. 19,622 người trong số đó khai báo list rỗng nên không mất gì. 8 người khai báo có anime nhưng không có dòng nào, và ta không biết vì sao (xem 2.2 d).
- **Giá trị `my_status`.** 537 dòng trong 80 triệu dòng mang một mã mà mô tả không định nghĩa. Ta không thể biết chúng nghĩa là gì nên chỉ báo cáo (xem 2.2 b).

**`my_status` được lưu thế nào.** Mỗi dòng của list table là một user và một anime, và `my_status` cho biết anime đó nằm ở đâu trong danh sách của user. File lưu một **con số**, không phải một từ. Các từ trong bảng dưới đây lấy từ mô tả của tác giả. *Tỉ lệ* là số dòng có mã đó chia cho 80,076,112.

| `my_status` | Ý nghĩa | Số dòng | Tỉ lệ |
|---|---|---|---|
| 2 | completed (đã xem xong) | 50,445,526 | 63.0% |
| 6 | plan to watch (dự định xem) | 19,392,104 | 24.2% |
| 1 | watching (đang xem) | 4,057,788 | 5.1% |
| 4 | dropped (bỏ dở) | 3,194,353 | 4.0% |
| 3 | on hold (tạm dừng) | 2,985,804 | 3.7% |
| 0, 5, 33, 55 | không rõ | 537 | dưới 0.001% |

Phần lớn các dòng (63.0%) là anime đã xem xong. Khoảng một phần tư (24.2%) chỉ là dự định: user chưa xem. Chỉ 12.8% là anime mà ai đó đang xem, đã bỏ dở hoặc tạm dừng.

**`my_score` được lưu thế nào.** Mô tả nói điểm từ 1 đến 10, nhưng file còn chứa cả 0. Số dòng có điểm từ 1 trở lên đúng bằng 46,358,322, tức bằng số "bản ghi có điểm" trong mô tả. Vậy **0 không có nghĩa là "user cho 0 điểm". Nó có nghĩa là "không chấm điểm".** Có 33,717,790 dòng như vậy (42.1%).

- **Vì sao điều này quan trọng (hãy chú ý).** Số 0 phải được đổi thành giá trị thiếu trước khi tính bất kỳ trung bình nào. Ví dụ: một user chấm hai anime 8 và 10, và để ba anime chưa chấm. Nếu giữ các số 0 thì trung bình là (8 + 10 + 0 + 0 + 0) / 5 = 3.6. Trung bình đúng của các điểm thật là 9.
- **Điểm 0 không có nghĩa là "đã xem nhưng không chấm"**, vì plan to watch thường nghĩa là chưa xem. Cách chia theo `my_status` nằm ở 2.2 a.
- **Trong các dòng có điểm, trung bình khoảng 7.49 và điểm nghiêng về phía cao.** 76.3% số dòng có điểm là từ 7 trở lên, chỉ 5.6% là từ 4 trở xuống, và điểm phổ biến nhất là 8 (11,394,082 dòng). Người ta chủ yếu chấm điểm cho thứ mình thích, nên điểm không phân bố đều từ 1 đến 10.
- **`title` không duy nhất** (2 anime trùng tiêu đề), vì vậy `anime_id` mới là key (xem 2.2 e).


In [3]:
# 1) Biến những gì khớp với mô tả thành assert
assert len(lists_raw) == 80_076_112
assert (lists_raw.my_score > 0).sum() == 46_358_322
assert lists_raw.my_score.between(0, 10).all()
assert anime_raw.anime_id.nunique() == 14_478
assert lists_raw.anime_id.nunique() == 14_478
assert anime_raw.anime_id.is_unique

# 2) Với mỗi status: bao nhiêu dòng chưa chấm điểm, và điểm trung bình là bao nhiêu?
print(lists_raw.groupby("my_status").my_score.agg(
    rows="size",
    unrated=lambda s: (s == 0).sum(),
    pct_unrated=lambda s: round((s == 0).mean() * 100, 1),
    mean_rated=lambda s: s[s > 0].mean(),
))

# 3) Các dòng có status không nằm trong mô tả
known_status = [1, 2, 3, 4, 6]
odd_status = lists_raw[~lists_raw.my_status.isin(known_status)]
print("\nodd status rows:", len(odd_status), "| users:", odd_status.username.nunique())
print(odd_status.head(8))

# 4) Username thiếu: đọc lại dạng văn bản thô để xem tên thật
username_text = pd.read_csv(paths["UserList.csv"], usecols=["username"],
                            dtype=str, keep_default_na=False)["username"]
print("\nusers_raw rows:", len(users_raw),
      "| username NaN:", users_raw.username.isna().sum(),
      "| matching raw text:", username_text[users_raw.username.isna()].tolist(),
      "| duplicated usernames:", users_raw.username.dropna().duplicated().sum())
print("lists username NaN:", lists_raw.username.isna().sum())

# 5) User không có dòng nào trong list table: list rỗng hay crawl bị thiếu?
list_users = set(lists_raw.username.cat.categories)
table_users = set(users_raw.username.dropna())
print("in lists, not in table:", len(list_users - table_users))
count_cols = ["user_watching", "user_completed", "user_onhold",
              "user_dropped", "user_plantowatch"]
users_no_list = users_raw[~users_raw.username.isin(list_users)]
declared = users_no_list[count_cols].sum(axis=1)
print("users without list:", len(users_no_list),
      "| declare 0 anime:", (declared == 0).sum(),
      "| declare > 0:", (declared > 0).sum())
print(declared[declared > 0].describe())

# 6) Key của list table; hai tập anime id có thật sự khớp nhau không?
print("\nduplicated (username, anime_id):",
      lists_raw.duplicated(["username", "anime_id"]).sum())
print("anime_id mismatch between tables:",
      len(set(lists_raw.anime_id) ^ set(anime_raw.anime_id)))

# 7) Các anime trùng tiêu đề
same_title = anime_raw[anime_raw.title.duplicated(keep=False)]
print("\nanime with a duplicated title:", len(same_title))
print(same_title.sort_values("title")[
    ["anime_id", "title", "type", "aired_string"]].head(10))

               rows   unrated  pct_unrated  mean_rated
my_status                                             
0               527       450         85.4    9.000000
1           4057788   2784664         68.6    7.952967
2          50445526   8046553         16.0    7.573541
3           2985804   2044638         68.5    7.267161
4           3194353   1687539         52.8    4.948901
5                 6         5         83.3    7.000000
6          19392104  19153938         98.8    7.852393
33                3         2         66.7    5.000000
55                1         1        100.0         NaN

odd status rows: 537 | users: 24
               username  anime_id  my_score  my_status
4452617       hatnclogs        21        10          0
4452623       hatnclogs       223        10          0
4452627       hatnclogs       249        10          0
4452644       hatnclogs      1604        10          0
4452726       hatnclogs     11061        10          0
4452790       hatnclogs     220

### 2.2 Kết luận từ lần chạy (giai đoạn EDA đầu tiên)

Các con số bên dưới lấy từ output của cell kiểm tra ở trên, chạy trên Kaggle với dữ liệu raw và safe load. Mỗi ý nói rõ ta đã đếm gì, con số đó nghĩa là gì, và ta chưa biết gì.

**a) Điểm 0 và status: ai thật sự chưa chấm điểm?**

Cách đọc bảng: *Chưa chấm* là số dòng có điểm 0 trong nhóm status đó. *% chưa chấm* là số đó chia cho số dòng của nhóm. *Điểm trung bình* chỉ tính trên các dòng có điểm.

| `my_status` | Ý nghĩa | Số dòng | Chưa chấm | % chưa chấm | Điểm trung bình (chỉ dòng có điểm) |
|---|---|---|---|---|---|
| 2 | completed | 50,445,526 | 8,046,553 | 16.0% | 7.57 |
| 1 | watching | 4,057,788 | 2,784,664 | 68.6% | 7.95 |
| 3 | on hold | 2,985,804 | 2,044,638 | 68.5% | 7.27 |
| 4 | dropped | 3,194,353 | 1,687,539 | 52.8% | 4.95 |
| 6 | plan to watch | 19,392,104 | 19,153,938 | 98.8% | 7.85 |

- **Phần lớn các dòng chưa chấm là "dự định xem", không phải điểm bị quên.** Tổng cộng có 33,717,790 dòng chưa chấm, và plan to watch chiếm 56.8% trong số đó. Điều này tự nhiên: không ai chấm điểm cho anime chưa xem (98.8% số dòng plan to watch chưa có điểm). Nếu bỏ plan to watch, tỉ lệ chưa chấm của các dòng còn lại là 24.0% (14,563,852 trên 60,684,008), chứ không phải 42.1%.
- **Điểm trung bình trên mọi dòng có điểm là 7.49.**
- **Người bỏ dở một anime chấm thấp hơn nhiều so với người xem xong:** 4.95 so với 7.57, chênh 2.62 điểm. Điều này chỉ mô tả dữ liệu. Ta không kiểm soát yếu tố nào nên không biết nguyên nhân.
- **Chỉ 47.2% số dòng dropped có điểm.** Con số 4.95 chỉ đến từ những người chọn chấm điểm sau khi bỏ dở, và họ có thể không giống tất cả những người đã bỏ dở. Hãy nhớ điều này với mọi câu hỏi về chất lượng anime.
- **Một điểm hơi lạ.** 238,166 dòng plan to watch (1.2% nhóm này) vẫn có điểm, trung bình 7.85. Có thể user đã đổi status sau khi chấm điểm, nhưng dữ liệu không cho biết, nên ta chỉ ghi nhận.
- **Bảng này giống nhau ở cả hai lần chạy** (lần đầu dùng mặc định của pandas, lần hai dùng safe load). Safe load chỉ sửa username. Bảng này đếm dòng theo status và không bao giờ dùng username, và 261 dòng của user `null` nằm trong bảng ở cả hai lần chạy (ví dụ 158 trong số 50,445,526 dòng completed).

**b) 537 dòng có status nằm ngoài mô tả (0, 5, 33, 55)**

Mô tả của tác giả chỉ định nghĩa 5 mã status: 1, 2, 3, 4, 6. Trong file có thêm 4 mã lạ.

| Mã | Số dòng | Chưa chấm | Điểm trung bình của các dòng có điểm |
|---|---|---|---|
| 0 | 527 | 450 (85.4%) | 9.00 (77 dòng có điểm) |
| 5 | 6 | 5 (83.3%) | 7.00 |
| 33 | 3 | 2 (66.7%) | 5.00 |
| 55 | 1 | 1 (100%) | không có điểm |

- **Chúng chỉ đến từ 24 user**, nên không phổ biến, và 98% số dòng mang mã 0. Ví dụ `hatnclogs` có vài dòng status 0 với điểm 10, còn `programmargorp` có các dòng status 0 không có điểm.
- **Điểm trung bình 9.00 của mã 0 không đáng tin.** Nó chỉ dựa trên 77 dòng, và các dòng in ra trong output cho thấy một user (`hatnclogs`) cho điểm 10 nhiều lần.
- 537 dòng chiếm chưa tới 0.001% của 80,076,112 dòng, nên ảnh hưởng lên bất kỳ thống kê nào gần như bằng 0.
- **Ta không biết các mã này nghĩa là gì.** Mô tả nói các giá trị khác là không rõ.
- **Đề xuất (nhóm quyết định):** loại các dòng này khỏi những phân tích có dùng `my_status` và báo cáo đã loại bao nhiêu dòng.

**c) Username `null`: một user có thật bị pandas đọc thành "thiếu"**

**Các thuật ngữ dùng trong mục này.**
- **User table** = `UserList.csv`. Một dòng là một user. Bảng có cột `username` và các cột về kích thước list của user theo khai báo trên profile (watching, completed, on hold, dropped, plan to watch).
- **List table** = `UserAnimeList.csv`. Một dòng là một anime trong list của một user. Cùng một username xuất hiện trong bao nhiêu dòng thì user đó có bấy nhiêu anime.
- Hai bảng được nối với nhau bằng `username`.

**1. Chuyện gì đã xảy ra.** Có **đúng một user có username là chữ `null`**. Trong user table đó là **một dòng**, vì `username` là duy nhất ở đó (302,675 dòng, 302,675 username khác nhau). Trong list table, cũng user này có **261 dòng**, mỗi dòng một anime.

Mặc định `pd.read_csv` biến các từ `null`, `NA`, `N/A`, `NaN`, `None` và ô trống thành giá trị thiếu. Đây là *normal load*. Với normal load, pandas đọc username `null` thành giá trị thiếu, nên:
- trong user table, user này trở thành "một dòng không có username";
- trong list table, 261 dòng của user này trở thành "261 dòng không có username";
- hai bên không nối được với nhau, vì một giá trị thiếu không khớp với gì cả.

**2. Cách sửa, gọi là safe load.** Ta đọc mọi file bằng `keep_default_na=False, na_values=[""]`. Cách này tắt danh sách từ mặc định của pandas, nên chỉ **ô trống** mới là giá trị thiếu, còn chữ `null` được giữ nguyên là văn bản. Các từ như `None` và `Unknown` cũng được giữ là văn bản, nên chính ta quyết định ở bước 3 từ nào nghĩa là "thiếu".

**3. Hai lần chạy của notebook này đặt cạnh nhau.** Lần đầu dùng normal load, lần hai dùng safe load.

| | Lần chạy 1 (normal load) | Lần chạy 2 (safe load) |
|---|---|---|
| Số username trong user table | 302,674 | **302,675** (khớp mô tả) |
| Username thiếu trong user table | 1 | **0** |
| Dòng list có username thiếu | 261 | **0** |
| User có ít nhất một dòng list | 283,044 | 283,045 |
| User không có dòng list nào | 19,631 | 19,630 |
| Trong đó, user khai báo hơn 0 anime | 9 | 8 |

**Dòng cuối của bảng nghĩa là gì.** Nó đếm những user có trong user table, khai báo có anime trên profile, nhưng **không có dòng nào trong list table**. Đây là một nhóm user được định nghĩa bằng cách so sánh hai bảng, không phải một bảng riêng. Ở lần chạy 1, user `null` nằm trong nhóm này, chỉ vì 261 dòng của họ mất username nên không ghép được. Ở lần chạy 2, các dòng được ghép đúng nên user rời khỏi nhóm: 9 thành 8. Các con số khác thay đổi vì cùng lý do: một user được tìm thấy, nên số user có dòng tăng một và số user không có dòng giảm một.

**4. Hai con số 261 khác nhau.** Số 261 xuất hiện ở cả hai bảng nhưng đếm hai thứ khác nhau:
- **User table:** một dòng, user `null`, với kích thước list khai báo là **261 anime**.
- **List table:** **261 dòng** có username `null`, mỗi dòng một anime.

Hai số này khớp nhau: profile nói 261 anime và list table có 261 dòng. Không có 261 user tên `null`.

**5. Những gì ta đã kiểm chứng (từ output trên Kaggle).**
- Phép kiểm tra tìm username thiếu trong `UserList.csv` in ra `[]`. Không còn username thiếu nào.
- `username` là duy nhất trong user table (`True`). `user_id` **không** duy nhất (302,673 giá trị khác nhau cho 302,675 dòng), nên ta không bao giờ nối bằng `user_id`.
- 261 dòng của user `null` mang các status bình thường (ví dụ 158 dòng có status 2, completed). Chúng là dữ liệu bình thường và không thuộc 537 dòng ở mục b.
- Bảng ở mục a không dùng username nên giống nhau ở cả hai lần chạy.

**6. Những gì ta CHƯA kiểm chứng.** Việc 261 = 261 là so sánh hai con số đếm cho một user. Đây là bằng chứng tốt cho thấy các dòng thuộc về user `null`, nhưng nó không chứng minh rằng phép nối chạy đúng cho mọi user. Còn hai kiểm tra chưa làm, cả hai nằm ở bước 3 (`kaggle/03_raw_quality_audit.ipynb`, chưa chạy):
- mọi username của list table đều có trong user table, và ngược lại (không có username "mồ côi" ở phía nào);
- với mọi user, kích thước list khai báo bằng số dòng trong list table.

Cho đến khi chạy xong, ta chỉ có thể nói user `null` đã được giải thích, chứ chưa thể nói phép nối bằng `username` là hoàn hảo.

**Quy tắc từ giờ:** mọi notebook đọc các bảng với `keep_default_na=False, na_values=[""]`.

**d) 19,630 user không có dòng nào trong list table**

- 19,622 người trong số đó (99.96%) khai báo 0 anime trong `UserList`. Họ có **list rỗng**, nên đây không phải dữ liệu crawl bị thiếu.
- **8 user khai báo có anime nhưng không có dòng nào.** Kích thước list khai báo của họ là 1 (nhỏ nhất), 146 (ở giữa, median) và 1,071 (lớn nhất), trung bình 289. Ở lần chạy 1 có 9 người: con số giảm xuống 8 khi `null` được đọc đúng, nên người thứ chín là user `null`, người thực ra có dòng. Với 8 người còn lại ta không biết lý do (list không crawl được, list private, hoặc user xóa list sau khi profile được crawl). Bước 3 sẽ kiểm tra xem những user này có các giá trị hoạt động khác không.
- Hệ quả: inner merge (chỉ giữ user có ở cả hai bảng) tự loại nhóm này. Nó không xóa dòng nào của list table.

**e) Key và định danh**

*Key* là một cột (hoặc một cặp cột) mà giá trị của nó xác định đúng một dòng.
- **Không có cặp `(username, anime_id)` nào lặp** (xác nhận lại trên toàn bộ 80,076,112 dòng). Vậy một dòng của list table đúng là một user và một anime.
- Hai tập `anime_id` (anime table và list table) giống hệt nhau (0 id khác nhau).
- `title` lặp đúng một lần: `Hinamatsuri` là anime 35102 (một OVA) và cũng là anime 36296 (một TV series bắt đầu phát sóng ngày 6 tháng 4 năm 2018). Chúng là hai anime khác nhau, nên `anime_id` vẫn là key và không bao giờ dùng `title` để nối bảng.

**Điều này có ý nghĩa gì với các câu hỏi nghiên cứu (giả thuyết, chưa kiểm chứng):**
- **Câu hỏi về việc bỏ dở:** tính trên các user đã bắt đầu xem (status 1, 2, 3, 4) và bỏ plan to watch ra khỏi mẫu số.
- **Câu hỏi về điểm:** nói rõ rằng điểm chỉ phản ánh những người chọn chấm điểm, và tỉ lệ dòng có điểm khác nhau rất nhiều giữa các nhóm status.


## 3. Khám phá chất lượng dữ liệu

**Bằng chứng đến từ đâu.** Việc kiểm tra chất lượng nằm trong [kaggle/02_column_dictionary_run.ipynb](../kaggle/02_column_dictionary_run.ipynb) (định dạng, placeholder, giá trị thiếu theo từng cột) và [kaggle/03_raw_quality_audit.ipynb](../kaggle/03_raw_quality_audit.ipynb) (placeholder, giá trị không thể xảy ra, mâu thuẫn giữa các bảng). Bước này chỉ đo đạc. Thứ tự của mọi bước nằm trong [docs/exploration_order.md](../docs/exploration_order.md).

**TODO:** Ghi lại dòng trùng / dòng rỗng, phân phối số / giá trị phi lý / outlier, tần suất của biến phân loại / nhãn không nhất quán, các mẫu dữ liệu thiếu và các mối quan hệ liên quan đến câu hỏi dự kiến. Giải thích vì sao mỗi vấn đề quan trọng, không chỉ in output.


In [4]:
# Ví dụ; chỉ chọn những kiểm tra có ý nghĩa sau khi đã đọc dữ liệu.
# display(missing_overview(raw_df))
# print("Duplicate rows:", raw_df.duplicated().sum())
# display(raw_df.describe(include="all").T)


## 4. Quyết định làm sạch, rồi lưu dataset đã xử lý

**TODO:** Với mỗi bước làm sạch, giải thích quy tắc và vì sao nó hợp lý. Dùng `report_row_change(step, before_df, after_df, reason)` cho việc lọc, xóa và nối bảng. Giữ nguyên dữ liệu raw gốc. Đừng mặc định rằng mọi giá trị thiếu hay outlier đều phải bị loại.

**Các quyết định còn mở** (giá trị phi lý, mã status không rõ, placeholder, anime chưa phát sóng, thiếu thông tin nhân khẩu, user không có list) được liệt kê trong [docs/exploration_order.md](../docs/exploration_order.md). Sau bước này, các thống kê của bước 5 ([kaggle/05a_verify_meanings.ipynb](../kaggle/05a_verify_meanings.ipynb) và hai notebook đi kèm) được chạy lại trên dữ liệu đã xử lý.


In [5]:
# Chỉ là mẫu ví dụ; thay bằng một thao tác có lý do rõ ràng.
# before_df = raw_df.copy()
# processed_df = before_df.copy()  # TODO: quy tắc làm sạch thật
# report_row_change("Step name", before_df, processed_df, "Explain the operation")
# processed_df.to_csv(ROOT / "data/processed/cleaned.csv", index=False)


## 5. Tóm tắt khám phá

Viết **3–5 quan sát dựa trên bằng chứng**, các vấn đề chất lượng nghiêm trọng, kế hoạch tiền xử lý và các hướng có thể cho câu hỏi nghiên cứu. Tách quan sát khỏi diễn giải. (Nhóm phải tự viết phần này.)
